# E11: Soporte espacial del clima

[Enunciado](../../talleres/E11.md) · [Entorno WSL](../../docs/instalacion-wsl.md)

Ejecutar en orden con el kernel del curso. Preparar las fuentes indicadas en el enunciado; no se descargan automáticamente. Las salidas se regeneran en `kit/salidas/`. Los originales se conservan.
[Linux/macOS](../../docs/instalacion-linux-macos.md). Ajustar la ruta explícita. Las operaciones se muestran directamente con pandas, GeoPandas y Rasterio; las funciones del kit quedan como referencia posterior de automatización.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# Editar esta ruta para que coincida con el clon local.
# Linux/macOS: Path.home() / "bigdata/big-data-postgraduate"
repositorio = Path("/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate")
KIT = repositorio / "kit"
RAW = KIT / "data/raw"
OUT = KIT / "salidas" / "E11"
assert KIT.is_dir(), "Corrige repositorio antes de continuar"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
from shapely.geometry import box, shape
from IPython.display import Image


## 1. Leer NASA y convertir sus parámetros en una tabla
Preparar NASA y CHIRPS según E11. El índice del diccionario es la fecha, que se convierte explícitamente. Sustituir -999 por faltante; nunca por lluvia cero. Guardar Parquet y volver a leer para verificar tipos y contenido.


In [ ]:
with open(RAW / 'nasa.json', encoding='utf-8') as archivo:
    nasa = json.load(archivo)
parametros = nasa['properties']['parameter']
diario = pd.DataFrame({'t2m': parametros['T2M'], 'lluvia_mm': parametros['PRECTOTCORR']})
diario = diario.rename_axis('fecha').reset_index()
diario['fecha'] = pd.to_datetime(diario['fecha'], format='%Y%m%d')
for columna in ['t2m', 'lluvia_mm']:
    diario[columna] = pd.to_numeric(diario[columna], errors='raise').replace(-999, float('nan'))
diario = diario.sort_values('fecha').reset_index(drop=True)
diario.to_parquet(OUT / 'nasa_diario.parquet', index=False)
pd.testing.assert_frame_equal(diario, pd.read_parquet(OUT / 'nasa_diario.parquet'))
control = {'nasa_dias': len(diario), 'nasa_temperatura_media': float(diario.t2m.mean()),
           'nasa_precipitacion_suma': float(diario.lluvia_mm.sum(min_count=1))}
assert control['nasa_dias'] == 31
assert abs(control['nasa_precipitacion_suma'] - 67.69) < 1e-8
display(diario.head())


### Recortar CHIRPS con una ventana explícita
La caja está en longitud/latitud. Transformarla al CRS del archivo, calcular la ventana y leer solo ese recorte. Conservar la máscara NoData y excluir negativos. Guardar la transformación de la ventana para ubicar correctamente el mapa.


In [ ]:
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
with rasterio.open(RAW / 'chirps.tif') as src:
    assert src.crs is not None
    bounds = transform_bounds('EPSG:4326', src.crs, -73.4, 5.5, -73.3, 5.6)
    ventana = from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
    lluvia = src.read(1, window=ventana, masked=True)
    lluvia = np.ma.masked_where(lluvia < 0, lluvia)
    transformacion = src.window_transform(ventana)
    crs_clima = src.crs
    control['chirps'] = {'crs': str(src.crs), 'nodata': src.nodata,
        'pixeles_validos': int(lluvia.count()), 'media_lluvia_mensual_mm': float(lluvia.mean()),
        'periodo': '2025-01', 'bbox': [-73.4, 5.5, -73.3, 5.6]}
assert lluvia.count() > 0
with open(OUT / 'control_clima.json', 'w', encoding='utf-8') as archivo:
    json.dump(control, archivo, ensure_ascii=False, indent=2)
display(control)


### Dibujar el punto NASA sobre el recorte CHIRPS


In [ ]:
punto = gpd.GeoDataFrame({'fuente': ['NASA POWER']}, geometry=[shape(nasa['geometry'])], crs=4326)
punto = punto.to_crs(crs_clima)
fig, ax = plt.subplots(figsize=(8, 6))
show(lluvia, transform=transformacion, ax=ax, cmap='Blues')
fig.colorbar(ax.images[0], ax=ax, label='Precipitación mensual (mm)')
punto.plot(ax=ax, color='red', markersize=35, label='Punto NASA')
ax.legend()
ax.set_title('CHIRPS, enero 2025: recorte demostrativo y punto NASA')
fig.savefig(OUT / 'mapa_clima.png', dpi=160, bbox_inches='tight')
plt.close(fig)
display(Image(filename=str(OUT / 'mapa_clima.png')))


## 2. Comprobar unidades y cobertura
CHIRPS se recorta a la caja indicada en el control, no a un municipio completo. Se conservan máscaras del archivo y se excluyen valores negativos; NASA trata -999 como faltante, no como cero. El punto rojo muestra la ubicación de la consulta NASA.

In [ ]:
resumen=pd.DataFrame([{'fuente':'NASA','soporte':'punto, suma temporal enero 2025','lluvia_mm':control['nasa_precipitacion_suma']},{'fuente':'CHIRPS','soporte':'recorte, media espacial mensual enero 2025','lluvia_mm':control['chirps']['media_lluvia_mensual_mm']}])
display(resumen)
resumen.to_csv(OUT/'comparacion_soportes_clima.csv',index=False)

## 3. Continuación del taller
La sección de eventos sigue en [E11](../../talleres/E11.md): ejecutar `python spark_taller.py eventos` desde `kit/`, con Java y Spark preparados según la guía WSL. Este notebook no ejecuta Spark ni sustituye la evidencia de watermark y eventos tardíos. Entregar mapa y comparación de soportes junto con esa evidencia.